# Трансформеры (архитектура)

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Трансформеры (архитектура)».

## Научная цель

Трансформер — архитектура нейронной сети на основе механизма внимания. При обработке каждого элемента последовательности модель сама решает, на какие другие элементы опереться, независимо от расстояния между ними. Это общий инструмент: на трансформерах работают модели для текста, последовательностей биомолекул, спектров и изображений.

В этом блокноте мы соберём компактный трансформер-энкодер с нуля на базовом PyTorch и обучим его задаче, где важны именно дальние связи: классифицировать короткие последовательности по правилу о далеко отстоящих элементах. Дополнительно мы извлечём и визуализируем карту внимания. Данные синтетические, блокнот исполняется на CPU за секунды.

## Интуиция за методом

Представьте, что вы читаете научную статью и пытаетесь понять сложное предложение в обсуждении. Вы не обрабатываете его слово за словом изолированно: вы мысленно обращаетесь к определению из введения, к формуле из методов, к рисунку из результатов. Ваш взгляд «прыгает» по тексту, связывая именно те места, которые помогают осмыслить текущий фрагмент.

Механизм **внимания** (attention) в трансформере делает то же самое: при обработке каждого элемента последовательности он вычисляет, насколько каждый другой элемент важен для него прямо сейчас — и взвешивает их вклад соответственно. Это происходит **одновременно для всех элементов** и **для любой пары**, независимо от расстояния между ними.

В отличие от рекуррентных сетей, где информация «телефонным проводом» тянется через все промежуточные шаги, трансформер связывает крайние элементы последовательности напрямую — как звонок по беспроводной связи.

**Ключевые термины, которые встретятся в блокноте:**

- **Токен** — минимальная единица входной последовательности (слово, символ, аминокислота, дискретное значение сигнала).
- **Эмбеддинг токена** — числовой вектор, в который токен переводится перед подачей в сеть; близкие по смыслу токены получают похожие векторы.
- **Позиционное кодирование** — вектор, прибавляемый к эмбеддингу и несущий информацию о порядковом номере позиции; без него трансформер воспринимает последовательность как множество без порядка.
- **Self-attention (само-внимание)** — операция, в которой каждый элемент взвешивает все остальные и формирует новое представление с учётом контекста.
- **Карта внимания** — матрица весов: строка i показывает, на какие позиции j обращает внимание позиция i.

## 1. Установка библиотек

В Google Colab большинство пакетов уже установлено. Ячейка ниже гарантирует наличие нужных библиотек. При локальном запуске она также корректна.

In [ ]:
%pip install -q torch==2.3.1 numpy matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Сгенерируем синтетическую задачу, специально созданную для того, чтобы **обнажить** то, чем трансформер отличается от других архитектур.

**Задача:** каждый объект — короткая последовательность токенов (целых чисел). Метка класса определяется тем, **совпадают ли первый и последний элементы** последовательности.

Это «дальняя зависимость»: между позицией 0 и позицией 15 находятся 14 шагов. Свёрточная сеть с малым окном такую пару не свяжет. Рекуррентная сеть должна «пронести» память через все промежуточные шаги. Трансформер связывает позиции 0 и 15 **напрямую** через механизм внимания.

In [ ]:
import numpy as np
import torch

torch.manual_seed(0)
rng = np.random.default_rng(0)

VOCAB = 12          # число различных токенов
SEQ = 16            # длина последовательности
N = 3000            # число примеров

X = rng.integers(0, VOCAB, size=(N, SEQ))
# Метка: 1, если первый и последний токены совпадают, иначе 0.
y = (X[:, 0] == X[:, -1]).astype('int64')
# Балансируем классы: совпадение редко, поэтому часть примеров правим.
for i in range(0, N, 2):
    X[i, -1] = X[i, 0]
y = (X[:, 0] == X[:, -1]).astype('int64')

n_tr = 2400
X_train, y_train = X[:n_tr], y[:n_tr]
X_test, y_test = X[n_tr:], y[n_tr:]
print(f'Обучающих: {len(X_train)}, тестовых: {len(X_test)}')
print(f'Доля положительного класса: {y.mean():.2f}')

## 4. Применение метода

### Шаг 1. Архитектура трансформера-энкодера

Соберём компактный трансформер из базовых блоков PyTorch. Пройдём по ключевым элементам:

1. **Эмбеддинг токенов** (`nn.Embedding`): каждый целый токен (0..VOCAB-1) переводится в вектор размерности `d_model`. Близкие по поведению токены после обучения получат похожие векторы.
2. **Позиционное кодирование** (`self.pos`): обучаемый тензор размера `(1, SEQ, d_model)`. Прибавляется к эмбеддингам — без него модель не знает, какой токен стоит первым, какой последним.
3. **TransformerEncoderLayer**: один блок внимания. Внутри — механизм self-attention (связывает все позиции попарно) + двухслойная полносвязная сеть + нормализация слоёв.
4. **Классификационная голова**: после двух таких блоков результаты агрегируются (усреднение по позициям) и подаются на линейный слой, дающий логиты двух классов.

In [ ]:
import torch.nn as nn


class TinyTransformer(nn.Module):
    """Компактный трансформер-энкодер для классификации последовательностей."""

    def __init__(self, vocab=VOCAB, seq=SEQ, d_model=32, n_heads=4):
        super().__init__()
        self.embed = nn.Embedding(vocab, d_model)
        # Позиционное кодирование — обучаемое, по одному вектору на позицию.
        self.pos = nn.Parameter(torch.randn(1, seq, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=64,
            dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.head = nn.Linear(d_model, 2)

    def forward(self, x):
        h = self.embed(x) + self.pos        # токены + позиции
        h = self.encoder(h)                 # self-attention блоки
        return self.head(h.mean(dim=1))     # агрегация по последовательности


model = TinyTransformer()
n_params = sum(p.numel() for p in model.parameters())
print(f'Параметров в модели: {n_params}')

### Шаг 2. Обучение

Ячейка ниже запускает стандартный цикл: батчи → CrossEntropyLoss → Adam. Заметьте, насколько быстро (за 15 эпох) трансформер достигает высокой точности на задаче с дальней зависимостью — именно потому, что внимание связывает позиции 0 и 15 напрямую.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

ds = TensorDataset(torch.from_numpy(X_train),
                   torch.from_numpy(y_train))
loader = DataLoader(ds, batch_size=64, shuffle=True)

opt = torch.optim.Adam(model.parameters(), lr=2e-3)
crit = nn.CrossEntropyLoss()

Xte = torch.from_numpy(X_test)
yte = torch.from_numpy(y_test)

history = {'loss': [], 'acc': []}
for epoch in range(1, 16):
    model.train()
    ep = 0.0
    for xb, yb in loader:
        opt.zero_grad()
        loss = crit(model(xb), yb)
        loss.backward()
        opt.step()
        ep += loss.item() * len(xb)
    model.eval()
    with torch.no_grad():
        acc = (model(Xte).argmax(1) == yte).float().mean().item()
    history['loss'].append(ep / len(X_train))
    history['acc'].append(acc)
    if epoch % 3 == 0 or epoch == 1:
        print(f'Эпоха {epoch:2d}: потеря {history["loss"][-1]:.4f}, точность на тесте {acc:.3f}')

### Шаг 3. Визуализация: ход обучения и карта внимания

Слева — динамика потерь и точности. Справа — **карта внимания** первого слоя для одного примера. Она показывает, на какие позиции смотрит модель при обработке каждой позиции: это ключевой «ага»-эксперимент блокнота.

In [ ]:
import matplotlib.pyplot as plt

# Извлечём веса внимания первого слоя энкодера для одного примера.
sample = Xte[:1]
model.eval()
with torch.no_grad():
    h = model.embed(sample) + model.pos
    attn_layer = model.encoder.layers[0].self_attn
    _, attn_w = attn_layer(h, h, h, need_weights=True,
                           average_attn_weights=True)
attn = attn_w[0].numpy()        # матрица (позиция, позиция)

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))
ep = range(1, len(history['loss']) + 1)
axes[0].plot(ep, history['loss'], color=VIZ['series'][0],
             label='Потеря (обучение)')
axes[0].plot(ep, history['acc'], color=VIZ['series'][1],
             label='Точность (тест)')
axes[0].set_title('Ход обучения')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Значение метрики')
axes[0].legend(loc='center right')

im = axes[1].imshow(attn, cmap='Blues')
axes[1].set_title('Карта внимания (первый слой)')
axes[1].set_xlabel('Позиция-источник')
axes[1].set_ylabel('Позиция-приёмник')
axes[1].grid(False)
fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04,
             label='Вес внимания')

fig.tight_layout()
plt.show()

### Как читать карту внимания

Карта внимания — это матрица размером SEQ × SEQ. Строка i показывает, **сколько внимания уделяет позиция i** каждой другой позиции j (по столбцам). Яркая клетка (i, j) означает, что при обработке позиции i модель сильно опирается на позицию j.

Что искать на этой карте:
- **Яркие клетки на крайних позициях (0-я строка и 0-й столбец, 15-я строка и 15-й столбец)** — признак того, что модель выучила сравнивать концы последовательности. Именно это и требовалось для нашей задачи.
- **Яркая диагональ** — модель смотрит «на себя»; это нейтральное поведение.
- **Однородная горизонтальная яркость** — модель не фокусируется ни на чём конкретном; это характерно для ранних эпох.

Помните: карта внимания — один из ракурсов на работу модели, но не исчерпывающее объяснение. Для строгой интерпретируемости используйте специальные методы (например, integrated gradients).

## 5. Интерпретация результата

- **Высокая точность на тесте** показывает, что трансформер уловил правило, связывающее первый и последний элементы. Локальная модель (например, свёрточная с малым окном) такую дальнюю зависимость без ухищрений не освоила бы.
- **Карта внимания** показывает, какие позиции модель связывает. Яркие клетки на пересечении крайних позиций означают, что модель научилась сопоставлять концы последовательности.
- Карты внимания **интерпретируют с осторожностью**: это не строгое объяснение решения, а одна из его составляющих.
- Без позиционного кодирования (`self.pos`) трансформер не различал бы порядок элементов и задачу бы не решил.

Стоимость стандартного внимания растёт квадратично с длиной последовательности; для длинного контекста применяют эффективные варианты внимания.

## 6. Попробуйте на своих данных

Трансформер-энкодер из этого блокнота подходит для классификации последовательностей дискретных элементов: символьных строк, последовательностей категорий, токенизированных сигналов.

1. Закодируйте свои последовательности целыми числами в диапазоне `0..VOCAB-1` и приведите к единой длине `SEQ` (дополнением или обрезкой).
2. Подготовьте метки классов `y`.
3. Снимите комментарии в ячейке ниже, задайте `VOCAB` и `SEQ`.
4. Для серьёзных задач берите предобученные трансформеры из библиотеки `transformers` — обучение с нуля требует много данных.

In [ ]:
# --- Шаблон подготовки своих последовательностей ---
# sequences = [...]              # список последовательностей
# labels = [...]                # метки классов
#
# VOCAB = ...                   # число различных элементов
# SEQ = ...                     # единая длина последовательности
# X = np.array(закодированные_и_выровненные, dtype='int64')
# y = np.array(labels, dtype='int64')
# Далее повторите создание и обучение модели из раздела 4.

## 7. Поэкспериментируйте сами

**Эксперимент 1. Роль позиционного кодирования.**
В классе `TinyTransformer` закомментируйте строку `self.pos = nn.Parameter(...)` и замените `h = self.embed(x) + self.pos` на `h = self.embed(x)`. Переобучите модель. Точность должна упасть: модель перестаёт различать порядок и не может решить задачу, опирающуюся на позиции 0 и 15.

**Эксперимент 2. Длина последовательности.**
Измените `SEQ = 16` на `SEQ = 32` и пересоздайте данные. Как меняется время обучения? Карта внимания теперь 32×32 — видны ли яркие клетки на угловых позициях 0 и 31?

**Эксперимент 3. Число голов внимания.**
В `TinyTransformer` измените `n_heads=4` на `n_heads=1`. Как изменилась карта внимания по структуре? Многоголовое внимание позволяет модели одновременно следить за несколькими типами зависимостей.

**Эксперимент 4. Посмотрите второй слой.**
В ячейке с картой внимания замените `model.encoder.layers[0]` на `model.encoder.layers[1]`. Сравните карты внимания первого и второго слоя: второй слой часто «смотрит» иначе.

## Краткие выводы

- Трансформер связывает любые два элемента последовательности **напрямую** через механизм внимания — без цепочки промежуточных шагов.
- Позиционное кодирование обязательно: без него модель не различает порядок элементов.
- Карта внимания позволяет увидеть, какие позиции модель считает связанными, — но интерпретировать её нужно осторожно, она не является полным объяснением предсказания.
- Трансформер обучается с нуля хуже при малом наборе данных; для научных задач предпочтительнее брать **предобученные модели** из `huggingface/transformers` и дообучать на своих данных.
- Сложность механизма внимания растёт квадратично с длиной последовательности; для длинных рядов используют эффективные варианты (FlashAttention, разрежённое внимание).